In [242]:
import pandas as pd
import numpy as np
import cobra
from configparser import ConfigParser
import working_w_seed_models as wm
from importlib import reload
reload(wm)

<module 'working_w_seed_models' from '/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py'>

In [ ]:
config = ConfigParser()
config.read("build_pbi_model.ini")
tmp = cobra.io.load_json_model("../../results/pbi_model_gapfill/bifermentans_gapfilled_from_cdiff_metabolomics_qc.json")
model = wm.Model(tmp, config)
model.model.objective = "bio1"

model.model.objective = "bio1"
if model.model.optimize().objective_value > 0:
    print("\n Model is able to produce Biomass!")
else:
    print("Model IS NOT able to produce Biomass")



/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:42: DtypeWarning: Columns (1,3,4,6,7,8,9,11,12,13,16,17,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  self.db = pd.read_csv(database_path)
/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1368: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  db_compound = pd.read_csv(compounds_filepath, sep = "\t", index_col = 0)


Length of conversions after 1st method:  721


/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1389: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  compounds = pd.read_csv(compounds_filepath, sep = "\t").set_index("name")


Length of conversions after 2nd method:  806
Length of conversions after 3rd method:  891

 Model is able to produce Biomass!


In [244]:
# Calculate minimal media to get max growth rate
met_ids = model.calculateMinimalMedia()
mets = [model.model.metabolites.get_by_id(i.replace("EX_", "")) for i in met_ids]
min_media = [i.name for i in mets]
min_media

Number of metabolites in minimal media:  40


['L-Glutamate [e0]',
 'L-Aspartate [e0]',
 'Phosphate [e0]',
 'L-Arginine [e0]',
 'L-Histidine',
 'L-Lysine [e0]',
 'L-Valine',
 'L-Proline',
 'L-Threonine',
 'Glycine [e0]',
 'L-Serine',
 'L-Isoleucine',
 'Nitrite',
 'Aminoethanol [e0]',
 'Adenosine [e0]',
 'Inosine [e0]',
 'D-Glucosamine [e0]',
 'D-Fructose [e0]',
 'Cytidine [e0]',
 'Glycerol-3-phosphate [e0]',
 'Cys-Gly [e0]',
 'Maltose [e0]',
 'Gly-Met [e0]',
 'Uridine [e0]',
 'Deoxycytidine [e0]',
 'Gly-Phe [e0]',
 'N-Acetyl-D-glucosamine [e0]',
 'Riboflavin [e0]',
 'D-Mannose [e0]',
 'gly-asp-L [e0]',
 'D-Glucose [e0]',
 'D-Ribose [e0]',
 'Gly-Leu [e0]',
 'gly-asn-L [e0]',
 'Deoxyuridine [e0]',
 'Deoxyguanosine [e0]',
 'Pantothenic acid [e0]',
 'H2O [e0]',
 'NH3 [e0]',
 'Niacin']

In [245]:
# remove double amino acids as options
import re

# Define the pattern to match metabolites with names like "Ala-His [e0]"
pattern1 = r'^[a-z]{3}-[A-Z]-[a-z]{3}-[A-Z] \[[a-z]\d\] exchange$'
pattern2 = r'^[A-Z]{1}[a-z]{2}-[A-Z]{1}[a-z]{2} \[[a-z]\d\] exchange$'
pattern3 = r'^[a-z]{3}-[a-z]{3}-[A-Z] \[[a-z]\d\] exchange$'
pattern4 = r'^[a-z]{3}-L-[A-Z][a-z]{2}-L \[[a-z]\d\] exchange$'


# Iterate through the boundary metabolites and remove those that match the pattern
for reaction in model.model.boundary:
    if re.match(pattern1, reaction.name) or re.match(pattern2, reaction.name) or re.match(pattern3, reaction.name) or re.match(pattern4, reaction.name):
        print(reaction.name)
        model.model.remove_reactions([reaction.id])


ala-L-Thr-L [e0] exchange
ala-L-glu-L [e0] exchange
Cys-Gly [e0] exchange
Gly-Met [e0] exchange
ala-L-asp-L [e0] exchange
Gly-Phe [e0] exchange
Gly-Cys [e0] exchange
Gly-Gln [e0] exchange
Gly-Tyr [e0] exchange
Ala-Leu [e0] exchange
gly-asp-L [e0] exchange
gly-pro-L [e0] exchange
Gly-Leu [e0] exchange
gly-glu-L [e0] exchange
Ala-Gln [e0] exchange
met-L-ala-L [e0] exchange
gly-asn-L [e0] exchange
Ala-His [e0] exchange


In [246]:

min_media_counts = {}
concentrations = list(np.arange(0.9, 1, 0.005))
for concentration in concentrations:
    met_ids = model.calculateMinimalMedia(concentration, minimize_components = True)
    mets = [model.model.metabolites.get_by_id(i.replace("EX_", "")) for i in met_ids]
    min_media = [i.name for i in mets]
    
    for metabolite in mets:
        if metabolite in min_media_counts:
            min_media_counts[metabolite] += 1
        else:
            min_media_counts[metabolite] = 1


import pandas as pd

# Convert the dictionary into a pandas DataFrame
min_media_df = pd.DataFrame(list(min_media_counts.items()), columns=['Metabolite ID', 'Count'])

# Add a column for the metabolite name
min_media_df['Metabolite Name'] = min_media_df['Metabolite ID'].apply(lambda x: model.model.metabolites.get_by_id(x.id).name)
min_media_df['Metabolite Formula'] = min_media_df['Metabolite ID'].apply(lambda x: model.model.metabolites.get_by_id(x.id).formula)

min_media_df.sort_values(by=["Count", "Metabolite Name"], ascending=[False, True], inplace=True)

min_media_df

Number of metabolites in minimal media:  14
Number of metabolites in minimal media:  14
Number of metabolites in minimal media:  14
Number of metabolites in minimal media:  15
Number of metabolites in minimal media:  16
Number of metabolites in minimal media:  16
Number of metabolites in minimal media:  17
Number of metabolites in minimal media:  18
Number of metabolites in minimal media:  19
Number of metabolites in minimal media:  20
Number of metabolites in minimal media:  21
Number of metabolites in minimal media:  22
Number of metabolites in minimal media:  23
Number of metabolites in minimal media:  24
Number of metabolites in minimal media:  25
Number of metabolites in minimal media:  27
Number of metabolites in minimal media:  28
Number of metabolites in minimal media:  30
Number of metabolites in minimal media:  32
Number of metabolites in minimal media:  33


,Metabolite ID,Count,Metabolite Name,Metabolite Formula
0,cpd00051_e0,20,L-Arginine [e0],C6H15N4O2
2,cpd00084_e0,20,L-Cysteine,C3H7NO2S
1,cpd00119_e0,20,L-Histidine,C6H9N3O2
8,cpd00322_e0,20,L-Isoleucine,C6H13NO2
4,cpd00107_e0,20,L-Leucine,C6H13NO2
3,cpd00060_e0,20,L-Methionine,C5H11NO2S
5,cpd00066_e0,20,L-Phenylalanine,C9H11NO2
7,cpd00129_e0,20,L-Proline,C5H9NO2
6,cpd00156_e0,20,L-Valine,C5H11NO2
9,cpd00179_e0,20,Maltose [e0],C12H22O11


In [248]:
min_media_df.index = ["EX_" + str(i) for i in min_media_df["Metabolite ID"].values]
for reaction_id in min_media_df.index:
    model.enableTransportReaction(reaction_id)
flux_df = model.model.optimize().fluxes
min_media_df['flux (mM) in 1 hr'] = -flux_df.loc[min_media_df.index]
min_media_df.to_csv("../../results/pbi_model_gapfill/minimal_media_counts.csv")

min_media_df

,Metabolite ID,Count,Metabolite Name,Metabolite Formula,flux (mM) in 1 hr
EX_cpd00051_e0,cpd00051_e0,20,L-Arginine [e0],C6H15N4O2,0.391249
EX_cpd00084_e0,cpd00084_e0,20,L-Cysteine,C3H7NO2S,0.009883
EX_cpd00119_e0,cpd00119_e0,20,L-Histidine,C6H9N3O2,0.228792
EX_cpd00322_e0,cpd00322_e0,20,L-Isoleucine,C6H13NO2,0.651178
EX_cpd00107_e0,cpd00107_e0,20,L-Leucine,C6H13NO2,0.300544
EX_cpd00060_e0,cpd00060_e0,20,L-Methionine,C5H11NO2S,0.300544
EX_cpd00066_e0,cpd00066_e0,20,L-Phenylalanine,C9H11NO2,0.058213
EX_cpd00129_e0,cpd00129_e0,20,L-Proline,C5H9NO2,1.750465
EX_cpd00156_e0,cpd00156_e0,20,L-Valine,C5H11NO2,0.219316
EX_cpd00179_e0,cpd00179_e0,20,Maltose [e0],C12H22O11,30.000000
